# Archelec — Topic Modeling & Semantic Mapping

Objectif immédiat :
1. Charger les métadonnées (`archelect_search.csv`)
2. Scanner les fichiers texte et construire un mapping `id -> chemin`
3. Joindre textes + métadonnées
4. Faire une première analyse exploratoire (qualité, longueurs, manquants)

### Imports des packages

In [1]:
import os
from pathlib import Path
import re

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42

#### 1) Chargement des données

In [2]:
base_dir = Path("..")  # depuis ntbks/
data_dir = base_dir / "data"

csv_path = data_dir / "archelect_search.csv"
text_dir = data_dir / "text_files"

csv_path, text_dir

(PosixPath('../data/archelect_search.csv'), PosixPath('../data/text_files'))

In [3]:
df = pd.read_csv(csv_path)
df.shape, df.columns

/var/folders/yw/_46mwd495f7cdxgzr0znl5jc0000gn/T/ipykernel_84858/928146548.py:1: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


((33030, 42),
 Index(['id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour',
        'cote', 'departement', 'departement-nom', 'departement-insee',
        'identifiant de circonscription', 'images', 'pdf', 'ocr_url',
        'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age',
        'titulaire-age-calcule', 'titulaire-age-tranche',
        'titulaire-profession', 'titulaire-mandat-en-cours',
        'titulaire-mandat-passe', 'titulaire-associations',
        'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste',
        'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom',
        'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule',
        'suppleant-age-tranche', 'suppleant-profession',
        'suppleant-mandat-en-cours', 'suppleant-mandat-passe',
        'suppleant-associations', 'suppleant-autres-statuts',
        'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations'],
       dtype='str'))

In [4]:
display(df.head(3))
df.info()

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,identifiant de circonscription,images,pdf,ocr_url,titulaire-nom,titulaire-prenom,titulaire-sexe,titulaire-age,titulaire-age-calcule,titulaire-age-tranche,titulaire-profession,titulaire-mandat-en-cours,titulaire-mandat-passe,titulaire-associations,titulaire-autres-statuts,titulaire-soutien,titulaire-liste,titulaire-decorations,suppleant-nom,suppleant-prenom,suppleant-sexe,suppleant-age,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations
0,EL009_L_1958_11_001_01_1_PF_01,1958-11-23,France;Élections législatives;Assemblée Nationale;Ve République,"Élections législatives de 1958, Ain - 01, circonscription n°01 : profession de foi de Emile Bouvard au tour 1",législatives,1,EL009,01,Ain,01 - Ain,1.0,https://ia903404.us.archive.org/8/items/EL009_L_1958_11_001_01_1_PF_01/EL009_L_1958_11_001_01_1_PF_01_000.jpg;https:...,https://ia903404.us.archive.org/8/items/EL009_L_1958_11_001_01_1_PF_01/EL009_L_1958_11_001_01_1_PF_01.pdf,https://ia903404.us.archive.org/8/items/EL009_L_1958_11_001_01_1_PF_01/EL009_L_1958_11_001_01_1_PF_01_djvu.txt,Bouvard,Emile,homme,non mentionné,non mentionné,non mentionné,industriel biscuitier,maire;conseiller général,non mentionné,non mentionné,non mentionné,Parti radical,non mentionné,non,Brayard,Joseph,homme,non mentionné,non mentionné,non mentionné,cultivateur,maire;conseiller général,non mentionné,non mentionné,non mentionné,Parti radical,non mentionné,non
1,EL009_L_1958_11_001_01_1_PF_02,1958-11-23,France;Ve République;Élections législatives;Assemblée Nationale,"Élections législatives de 1958, Ain - 01, circonscription n°01 : profession de foi de Albert Jouvent au tour 1",législatives,1,EL009,01,Ain,01 - Ain,1.0,https://ia803405.us.archive.org/15/items/EL009_L_1958_11_001_01_1_PF_02/EL009_L_1958_11_001_01_1_PF_02_000.jpg;https...,https://ia803405.us.archive.org/15/items/EL009_L_1958_11_001_01_1_PF_02/EL009_L_1958_11_001_01_1_PF_02.pdf,https://ia803405.us.archive.org/15/items/EL009_L_1958_11_001_01_1_PF_02/EL009_L_1958_11_001_01_1_PF_02_djvu.txt,Jouvent,Albert,homme,non mentionné,non mentionné,non mentionné,exploitant forestier,non mentionné,non mentionné,groupes de pression,non mentionné,Union pour la nouvelle République,non mentionné,oui,Vourlat,Noël,homme,non mentionné,non mentionné,non mentionné,cultivateur,conseiller municipal,non mentionné,non mentionné,prisonnier de guerre,Union pour la nouvelle République,non mentionné,non
2,EL009_L_1958_11_001_01_1_PF_03,1958-11-23,Élections législatives;France;Assemblée Nationale;Ve République,"Élections législatives de 1958, Ain - 01, circonscription n°01 : profession de foi de Emile Machurat au tour 1",législatives,1,EL009,01,Ain,01 - Ain,1.0,https://ia903406.us.archive.org/33/items/EL009_L_1958_11_001_01_1_PF_03/EL009_L_1958_11_001_01_1_PF_03_000.jpg;https...,https://ia903406.us.archive.org/33/items/EL009_L_1958_11_001_01_1_PF_03/EL009_L_1958_11_001_01_1_PF_03.pdf,https://ia903406.us.archive.org/33/items/EL009_L_1958_11_001_01_1_PF_03/EL009_L_1958_11_001_01_1_PF_03_djvu.txt,Machurat,Emile,homme,non mentionné,non mentionné,non mentionné,ouvrier,non mentionné,non mentionné,politique,résistant,Parti communiste français,non mentionné,non,Bozonnet,Georges,homme,non mentionné,non mentionné,non mentionné,cultivateur,non mentionné,non mentionné,non mentionné,non mentionné,Parti communiste français,non mentionné,non


<class 'pandas.DataFrame'>
RangeIndex: 33030 entries, 0 to 33029
Data columns (total 42 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   id                              33030 non-null  str   
 1   date                            33030 non-null  str   
 2   subject                         33030 non-null  str   
 3   title                           33030 non-null  str   
 4   contexte-election               33030 non-null  str   
 5   contexte-tour                   33030 non-null  int64 
 6   cote                            33030 non-null  str   
 7   departement                     32688 non-null  str   
 8   departement-nom                 32669 non-null  str   
 9   departement-insee               32666 non-null  str   
 10  identifiant de circonscription  32136 non-null  object
 11  images                          33003 non-null  str   
 12  pdf                             31855 non-null  str   
 1

### 2) Nettoyage minimal des métadonnées

On normalise :
- noms de colonnes
- types (date, âge)

In [5]:
# standardisation des noms de colonnes
df.columns = [c.strip().lower() for c in df.columns]

print(df.shape)

# conversion de la colonne "date" en datetime et extraction de l'année

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["year"] = df["date"].dt.year

df[["date", "year"]].head()

# conversion de la colonne "titulaire-age-calcule" en numérique, en gérant les erreurs
age_col = "titulaire-age-calcule"
if age_col in df.columns:
    df[age_col] = pd.to_numeric(df[age_col], errors="coerce")

(33030, 42)


In [6]:
# Etant donné qu'on a que les données legislatives de 1981, 1988 et 1993, on 
# garde que ces années pour l'instant et également que les legislatives

df_legislatives = df.copy()
df_legislatives = df_legislatives[df_legislatives['contexte-election']!= 'présidentielle']
df_legislatives = df_legislatives[df_legislatives['year'].isin([1981, 1988, 1993])]
print(df_legislatives.shape)

(12498, 43)


### 3) Scanner les fichiers texte et construire `id -> path`

Les fichiers sont de forme :
`EL174_L_1988_06_001_01_1_PF_03.txt`

On construit un dictionnaire :
`id_to_path["EL174_L_1988_06_001_01_1_PF_03"] = ".../1988/legislatives/EL174_L_1988_06_001_01_1_PF_03.txt"`

In [17]:
# construction d'un dictionnaire id -> path pour les fichiers texte
# parcourt tous les fichiers texte dans le répertoire text_dir et crée un 
# dico où la clé est l'identifiant du fichier (le nom du fichier sans l'extension) 
# et la valeur est le chemin complet vers ce fichier. 

def build_id_to_path(text_root: Path) -> dict:
    id_to_path = {}
    for p in text_root.rglob("*.txt"):
        file_id = p.stem  # nom sans extension
        id_to_path[file_id] = p
    return id_to_path

id_to_path = build_id_to_path(text_dir)

len(id_to_path), list(id_to_path.items())[:2]

(12746,
 [('EL190_L_1993_03_016_02_2_PF_02',
   PosixPath('../data/text_files/EL190_L_1993_03_016_02_2_PF_02.txt')),
  ('EL175_L_1988_06_037_03_1_PF_03',
   PosixPath('../data/text_files/EL175_L_1988_06_037_03_1_PF_03.txt'))])

In [18]:
ID_COL = "id"  # adapte si nécessaire
assert ID_COL in df_legislatives.columns, f"Colonne {ID_COL} introuvable dans le CSV"

# normaliser l'ID (string, sans espaces)
df_legislatives[ID_COL] = df_legislatives[ID_COL].astype(str).str.strip()

df_legislatives[ID_COL].head()

19480    EL134_L_1981_06_001_01_1_PF_01
19481    EL134_L_1981_06_001_01_1_PF_02
19482    EL134_L_1981_06_001_01_1_PF_03
19483    EL134_L_1981_06_001_01_1_PF_04
19484    EL134_L_1981_06_001_01_1_PF_05
Name: id, dtype: str

In [20]:
df_legislatives["text_path"] = df_legislatives[ID_COL].map(id_to_path)

missing_rate = df_legislatives["text_path"].isna().mean()
missing_rate

np.float64(8.001280204832773e-05)

In [21]:
missing = df_legislatives[df_legislatives["text_path"].isna()][ID_COL].head(20).tolist()
display(missing)
print(len(df_legislatives[df_legislatives["text_path"].isna()][ID_COL].tolist()))

['EL177_L_1988_06_094_04_1_PF_01']

1


### 4) Charger les textes

On charge uniquement les lignes avec `text_path` trouvé. Puis on mesure la longueur (nombre de tokens simples = split sur espaces).

In [22]:
# fonction pour charger le contenu d'un fichier texte à partir de son chemin

def load_text_file(path: Path) -> str | None:
    if path is None or (isinstance(path, float) and np.isnan(path)):
        return None
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        # fallback si certains fichiers sont en latin-1
        try:
            return path.read_text(encoding="latin-1")
        except Exception:
            return None
    except Exception:
        return None

df_legislatives["text"] = df_legislatives["text_path"].apply(load_text_file)

df_legislatives["text_length"] = df_legislatives["text"].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)

df_legislatives[["text_path", "text_length", "text"]].head(3)

,text_path,text_length,text
19480,../data/text_files/EL134_L_1981_06_001_01_1_PF_01.txt,552,ELECTIONS LEGISLATIVES - 14 JUIN 1981 AIN 1e CIRCONSCRIPTION\nMicheline ANTONUCCI\nPOUR QUE ÇA DURE ...\nLa victoire...
19481,../data/text_files/EL134_L_1981_06_001_01_1_PF_02.txt,642,"Sciences Po / fonds CEVIPOF\nELECTIONS LÉGISLATIVES - 14 JUIN 1981\n1"" circonscription de l'Ain\nMarcel BENOIT\nMair..."
19482,../data/text_files/EL134_L_1981_06_001_01_1_PF_03.txt,644,Sciences Po / fonds CEVIPOF\nRÉPUBLIQUE FRANÇAISE - LIBERTÉ - ÉGALITÉ - FRATERNITÉ - DÉPARTEMENT DE L'AIN\n0\nPARTI ...


In [23]:
print("Docs total:", len(df_legislatives))
print("Docs avec texte:", (df_legislatives["text_length"] > 0).sum())
print("Taux texte manquant:", (df_legislatives["text_length"] == 0).mean())

df_legislatives["text_length"].describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])

Docs total: 12498
Docs avec texte: 12497
Taux texte manquant: 8.001280204832773e-05


count    12498.000000
mean       634.657305
std        318.065124
min          0.000000
5%         252.000000
10%        269.000000
25%        403.000000
50%        563.000000
75%        836.000000
90%       1063.000000
95%       1261.000000
max       3232.000000
Name: text_length, dtype: float64

In [26]:
df_test = df_legislatives[df_legislatives["year"].isin([1981, 1988, 1993])]

print("Docs total:", len(df_legislatives))
print("Docs avec texte:", (df_legislatives["text_length"] > 0).sum())
print("Taux texte manquant:", (df_legislatives["text_length"] == 0).mean())

Docs total: 12498
Docs avec texte: 12497
Taux texte manquant: 8.001280204832773e-05
